In [1]:
import hashlib
import json
import math
import re
import textwrap
import unicodedata
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RNG = np.random.default_rng(20260813)   # fixed seed: every permutation/bootstrap is reproducible

# ---------------------------------------------------------------- palette
# Validated categorical palette. Slots are assigned in this order and never cycled;
# a chart needing more than 8 identities folds the tail into "Other" instead.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

SURFACE   = "#fcfcfb"   # chart surface, painted explicitly
INK       = "#0b0b0b"   # primary ink
INK_2     = "#52514e"   # secondary ink
MUTED     = "#898781"   # axis labels, annotations
GRID      = "#e1e0d9"   # hairline gridlines
AXIS      = "#c3c2b7"   # baseline
ZERO_GREY = "#d8d7d1"   # bars for categories with zero observations

# Sequential (magnitude) and diverging (polarity) ramps, built from the same blue ramp.
BLUES = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
         "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ  = LinearSegmentedColormap.from_list("seq_blue", BLUES)
DIV  = LinearSegmentedColormap.from_list("div_blue_red", ["#0d366b", "#3987e5", "#f0efec",
                                                          "#e34948", "#8f1f1f"])

mpl.rcParams.update({
    "figure.facecolor": SURFACE,   "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,  "figure.dpi": 110,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"],
    "font.size": 9.5,
    "text.color": INK,             "axes.labelcolor": INK_2,
    "xtick.color": MUTED,          "ytick.color": MUTED,
    "xtick.labelsize": 8.5,        "ytick.labelsize": 8.5,
    "axes.edgecolor": AXIS,        "axes.linewidth": 0.8,
    "axes.spines.top": False,      "axes.spines.right": False,
    "axes.grid": True,             "grid.color": GRID,
    "grid.linewidth": 0.7,         "axes.axisbelow": True,
    "legend.frameon": False,       "legend.fontsize": 8.5,
    "axes.titlesize": 11,          "axes.titleweight": "semibold",
    "axes.titlelocation": "left",  "axes.titlepad": 14,
})

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)

print("style loaded | palette:", " ".join(SERIES))

style loaded | palette: #2a78d6 #eb6834 #1baf7a #eda100 #e87ba4 #008300 #4a3aa7 #e34948


In [2]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

# Works whether the notebook is run from src/ or from the repository root.
REPO = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

RUN_PATH = REPO / "runs" / "predictions_chr_sample25.jsonl"
CORPUS_PATH = REPO / "data" / "corpus_chr.jsonl"
#XLSX_PATH = REPO / "runs" / "gide_opus5_relations.xlsx"
IMGS_PATH = REPO / "imgs"

CLOSED_FIELDS = [
    "relation_type", "structure",
    "status", "reciprocity", "exclusivity", "adultery",
    "explicitness", "explicitness_in_plot",
    "centrality", "framing", "outcome",
    "confidence",
]
REL_COLUMNS = ["novel_id","orientation", "model", "relation_id", "characters"] + CLOSED_FIELDS + ["n_evidence", "comment"]
EV_COLUMNS = ["novel_id", "relation_id", "evidence_index", "quote", "locator", "description"]

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)

In [3]:
records = [
    json.loads(line)
    for line in RUN_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{len(records)} record(s)\n")

100 record(s)



In [4]:
def relation_rows(records):
    for r in records:
        annotation = r.get("annotation") or {}
        for rel in annotation.get("relations") or []:
            row = {
                "novel_id": r.get("novel_id"),
                'orientation': r.get("orientation"),
                "model": r.get("model"),
                "relation_id": rel.get("relation_id"),
                "characters": " | ".join(str(c) for c in rel.get("characters") or []),
                "n_evidence": len(rel.get("evidence") or []),
                "comment": rel.get("comment"),
            }
            row.update({f: rel.get(f) for f in CLOSED_FIELDS})
            yield row


relations = pd.DataFrame(list(relation_rows(records)), columns=REL_COLUMNS)
relations

,novel_id,orientation,model,relation_id,characters,relation_type,structure,status,reciprocity,exclusivity,adultery,explicitness,explicitness_in_plot,centrality,framing,outcome,confidence,n_evidence,comment
0,alanic_roses,hetero,anthropic/claude-opus-5,R1,Estelle Gerfaux | Vincent Marcenat,female_male,dyadic,married,reciprocal,exclusive,loyal,explicit,explicit,main,positive,stable_or_open,high,7,"Relation centrale. Le bond initial est un mariage de dévouement, p..."
1,alanic_roses,hetero,anthropic/claude-opus-5,R2,Estelle Gerfaux | Renaud Jonchère,female_male,dyadic,engaged,asymmetrical,exclusive,adultery_one,explicit,hidden_except,main,neutral_ambivalent,separation_erasure,high,7,"Fiançailles secrètes, non officialisées mais nommées comme telles ..."
2,alanic_roses,hetero,anthropic/claude-opus-5,R3,Adrien Gerfaux | Monique Françon,female_male,dyadic,married,reciprocal,exclusive,loyal,explicit,explicit,secondary,positive,stable_or_open,high,4,"Sous-intrigue conjugale entièrement valorisée, contrepoint heureux..."
3,alanic_roses,hetero,anthropic/claude-opus-5,R4,Vincent Marcenat | Odette de Tintaniac (Mme Marcenat),female_male,dyadic,married,asymmetrical,exclusive,unclear,explicit,explicit,secondary,moralized_pathologized,fatal_ending,high,4,Mariage malheureux servant de préhistoire au roman. La règle affic...
4,alanic_roses,hetero,anthropic/claude-opus-5,R5,Renaud Jonchère | Mme de Leucate,female_male,dyadic,physical_relationship,unclear,adultery,external_adultery,rumor,explicit,secondary,eroticized_exoticized,separation_erasure,medium,3,"Relation connue seulement par lettre et chroniques (« sirène », « ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,zuylen_chemin,lesbiennes,anthropic/claude-opus-5,R10,Giovanna Montenari | le fils aîné du roi de France,female_male,dyadic,romantic_relationship,reciprocal,adultery,external_adultery,explicit,explicit_except,episode,decadent_sublime,fatal_ending,high,3,Explicit adulterous relation (exclusivity = adultery with adultery...
898,zuylen_chemin,lesbiennes,anthropic/claude-opus-5,R11,Angiola Montenari | le prétendant qui répudia Donna Fiamma,female_male,triangular,seduction,one_sided,uncommitted,none,explicit,explicit,episode,decadent_sublime,fatal_ending,medium,3,"Coded as triangular because the repudiated Donna Fiamma, of a 'lig..."
899,zuylen_chemin,lesbiennes,anthropic/claude-opus-5,R12,Andriana Montenari | l'auteur inconnu des lettres d'amour | Don Fe...,female_male,triangular,romantic_relationship,unclear,adultery,external_adultery,explicit,hidden,episode,decadent_sublime,fatal_ending,low,2,"Only two sentences of ancestral legend, but they name love letters..."
900,zuylen_chemin,lesbiennes,anthropic/claude-opus-5,R13,Dante Alighieri | Donna Béatrice,female_male,dyadic,romantic_relationship,one_sided,uncommitted,none,explicit,explicit,episode,decadent_sublime,separation_erasure,medium,2,"An embedded literary legend, retained because it is the explicit m..."


In [8]:
relations.columns

Index(['novel_id', 'orientation', 'model', 'relation_id', 'characters', 'relation_type', 'structure', 'status', 'reciprocity', 'exclusivity', 'adultery', 'explicitness', 'explicitness_in_plot', 'centrality', 'framing',
       'outcome', 'confidence', 'n_evidence', 'comment'],
      dtype='object')

In [50]:
relations.reciprocity.value_counts()

reciprocity
asymmetrical    288
reciprocal      232
one_sided       207
unclear         175
Name: count, dtype: int64

In [69]:
import pandas as pd


# ------------------------------------------------------------------
# 1. Column selection
# ------------------------------------------------------------------
ACTIVE_COLS = ["relation_type", "status", "reciprocity", 
               "exclusivity","framing","outcome"]  # build the axes
SUPP_COLS   = []                                           # projected only, not active

# ------------------------------------------------------------------
# 2. Modality renaming: {column: {old_value: new_label}}
#    Values not listed are left untouched.
#    Mapping two old values to the same new label merges them.
# ------------------------------------------------------------------
RENAME_MODALITIES = {
    # "education": {
    #     "hs":  "NoDegree",
    #     "bac": "Bachelor",
    #     "mst": "Master",
    #     "phd": "PhD",
    # },
    # "satisfaction": {
    #     "1": "VeryLow",
    #     "2": "Low",
    #     "3": "Medium",
    #     "4": "High",
    #     "5": "VeryHigh",
    # },
    # "age_group": {
    #     "18_25": "18-25",
    #     "26_40": "26-40",
    #     "41_60": "41-60",
    #     "60p":   "60+",
    # },
    # "region": {"N": "North", "S": "South", "E": "East", "W": "West"},
}

RENAME_COLUMNS = {}#{"satisfaction": "satisf"}  # optional, shortens plot labels

# ------------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------------
def prepare_for_mca(
    data,
    cols,
    rename_modalities=None,
    rename_columns=None,
    na_label="Missing",
    min_freq=0.02,        # set to 0 to disable rare-modality grouping
    other_label="Other",
    prefix_columns=False, # prince already prefixes as "col_modality"
):
    X = data[cols].copy()

    # everything to clean strings
    for c in cols:
        X[c] = X[c].astype("string").str.strip()
        X[c] = X[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    # rename modalities
    if rename_modalities:
        for c, mapping in rename_modalities.items():
            if c in X.columns:
                X[c] = X[c].replace(mapping)

    # missing values -> explicit modality (MCA cannot take NaN)
    X = X.fillna(na_label)

    # group rare modalities
    if min_freq:
        for c in cols:
            freq = X[c].value_counts(normalize=True)
            rare = freq[freq < min_freq].index
            if len(rare):
                print(f"[{c}] grouped into '{other_label}': {list(rare)}")
                X[c] = X[c].mask(X[c].isin(rare), other_label)

    if prefix_columns:
        for c in cols:
            X[c] = c + "_" + X[c]

    if rename_columns:
        X = X.rename(columns=rename_columns)

    return X.astype({c: "category" for c in X.columns})


X_active = prepare_for_mca(relations, ACTIVE_COLS, RENAME_MODALITIES, RENAME_COLUMNS, min_freq=0.05)
X_supp   = prepare_for_mca(relations, SUPP_COLS,   RENAME_MODALITIES, RENAME_COLUMNS, min_freq=0.0)


[relation_type] grouped into 'Other': ['mixed_gender', 'queer', 'unknown_gender', 'unclear']
[status] grouped into 'Other': ['engaged']
[exclusivity] grouped into 'Other': ['primary_relationship', 'secondary_relationship', 'polyamory', 'emotionally_exclusive']
[framing] grouped into 'Other': ['redemptive_spiritualized']
[outcome] grouped into 'Other': ['social_disgrace', 'illness_madness_decline']


In [70]:
X_active

,relation_type,status,reciprocity,exclusivity,framing,outcome
0,female_male,married,reciprocal,exclusive,positive,stable_or_open
1,female_male,Other,asymmetrical,exclusive,neutral_ambivalent,separation_erasure
2,female_male,married,reciprocal,exclusive,positive,stable_or_open
3,female_male,married,asymmetrical,exclusive,moralized_pathologized,fatal_ending
4,female_male,physical_relationship,unclear,adultery,eroticized_exoticized,separation_erasure
...,...,...,...,...,...,...
897,female_male,romantic_relationship,reciprocal,adultery,decadent_sublime,fatal_ending
898,female_male,seduction,one_sided,uncommitted,decadent_sublime,fatal_ending
899,female_male,romantic_relationship,unclear,adultery,decadent_sublime,fatal_ending
900,female_male,romantic_relationship,one_sided,uncommitted,decadent_sublime,separation_erasure


In [71]:
import numpy as np, pandas as pd, plotly.graph_objects as go

PALETTE = ["#4C78A8","#F58518","#54A24B","#E45756","#72B7B2",
           "#B279A2","#EECA3B","#FF9DA6","#9D755D","#BAB0AC"]

def _dummies(X):
    """Indicator matrix + (variable, modality) bookkeeping, built column by
    column so names containing '_' are never mis-parsed."""
    frames, var_of, mod_of = [], [], []
    for col in X.columns:
        s = X[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.cat.remove_unused_categories()
        d = pd.get_dummies(s.astype(str), dtype=float)
        var_of += [col] * d.shape[1]; mod_of += list(d.columns)
        d.columns = [f"{col}_{m}" for m in d.columns]
        frames.append(d)
    return pd.concat(frames, axis=1), var_of, mod_of

class MCA:
    """MCA on the indicator matrix via SVD of the standardised residuals.
    eig sums to K/Q-1 | contrib sums to 100 per axis | cos2 sums to 1 per point."""

    def __init__(self, X, n_components=5):
        Z, var_of, mod_of = _dummies(X)
        n, K = Z.shape; Q = X.shape[1]; max_dim = K - Q
        S = int(min(n_components, max_dim))

        N = Z.to_numpy(); P = N / N.sum()
        r = P.sum(1)                     # row masses = 1/n
        c = P.sum(0)                     # col masses = n_k/(nQ)
        Sres = (P - np.outer(r, c)) / np.sqrt(np.outer(r, c))
        U, sv, Vt = np.linalg.svd(Sres, full_matrices=False)
        eig_all = (sv ** 2)[:max_dim]    # full spectrum, needed by Greenacre
        U, sv, Vt = U[:, :S], sv[:S], Vt[:S]; eig = sv ** 2
        dims = [f"Dim{i+1}" for i in range(S)]

        F = (U * sv) / np.sqrt(r)[:, None]        # individuals, principal coords
        G = (Vt.T * sv) / np.sqrt(c)[:, None]     # modalities, principal coords
        # exact chi2 distances to centroid -> exact cos2 even with few axes
        d2_rows = (((P / r[:, None] - c) ** 2) / c).sum(1)
        d2_cols = (((P / c).T - r) ** 2 / r).sum(1)

        self.n, self.Q, self.K, self.dims = n, Q, K, dims
        self.row_coord   = pd.DataFrame(F, X.index, dims)
        self.col_coord   = pd.DataFrame(G, Z.columns, dims)
        self.row_contrib = pd.DataFrame(r[:, None] * F**2 / eig * 100, X.index, dims)
        self.col_contrib = pd.DataFrame(c[:, None] * G**2 / eig * 100, Z.columns, dims)
        self.row_cos2    = pd.DataFrame(F**2 / d2_rows[:, None], X.index, dims)
        self.col_cos2    = pd.DataFrame(G**2 / d2_cols[:, None], Z.columns, dims)
        self._F, self._eig = F, eig

        # Benzécri correction with Greenacre's adjusted total.
        # These sum to LESS than 100% by construction — that's the point, not a bug.
        tot = K / Q - 1
        benz = np.where(eig > 1/Q, ((Q/(Q-1)) * (eig - 1/Q))**2, 0.0)
        green = (Q/(Q-1)) * ((eig_all**2).sum() - (K-Q)/Q**2)
        self.eig = pd.DataFrame({
            "eigenvalue": eig, "%_inertia": eig/tot*100,
            "cum_%": np.cumsum(eig/tot*100), "benzecri_eig": benz,
            "benzecri_%": benz/green*100,
            "benzecri_cum_%": np.cumsum(benz/green*100)}, index=dims)  # keep unrounded

        self.col_info = pd.DataFrame(
            {"variable": var_of, "modality": mod_of,
             "count": N.sum(0).astype(int), "mass": c}, index=Z.columns)

    def project_supplementary(self, X_supp):
        """Illustrative variables, via the transition formula."""
        Zs, _, _ = _dummies(X_supp); A = Zs.to_numpy()
        G = (A / A.sum(0)).T @ self._F / np.sqrt(self._eig)
        return pd.DataFrame(G, Zs.columns, self.dims)

    def axis_label(self, a):
        e = self.eig.loc[f"Dim{a}"]
        return f"Dim {a} — {e['%_inertia']:.1f}% (Benzécri {e['benzecri_%']:.1f}%)"

In [72]:
def top_contributions(res, axis=1, k=15):
    d = f"Dim{axis}"
    out = res.col_info.copy()
    out["coord"]     = res.col_coord[d]
    out["contrib_%"] = res.col_contrib[d]
    out["cos2"]      = res.col_cos2[d]
    out["pole"]      = np.where(out["coord"] >= 0, "+", "-")
    out["above_avg"] = out["contrib_%"] > 100 / res.K     # uniform threshold
    return out.sort_values("contrib_%", ascending=False).head(k).round(3)[
        ["variable","modality","count","coord","contrib_%","cos2","pole","above_avg"]]

def top_cos2(res, axes=(1,2), k=15):
    dims = [f"Dim{a}" for a in axes]
    out = res.col_info.copy()
    for d in dims: out[f"cos2_{d}"] = res.col_cos2[d]
    out["cos2_plane"] = res.col_cos2[dims].sum(1)
    out["quality"]    = np.sqrt(out["cos2_plane"])        # = radius on the circle
    return out.sort_values("cos2_plane", ascending=False).head(k).round(3)[
        ["variable","modality","count"] + [f"cos2_{d}" for d in dims]
        + ["cos2_plane","quality"]]

def variable_summary(res, axes=(1,2)):
    dims = [f"Dim{a}" for a in axes]
    g = res.col_info[["variable"]].copy()
    for d in dims: g[f"ctr_{d}"] = res.col_contrib[d]
    g["cos2_plane"] = res.col_cos2[dims].sum(1)
    return g.groupby("variable").agg(
        n_modalities=("variable","size"),
        **{f"ctr_{d}_%": (f"ctr_{d}","sum") for d in dims},
        mean_cos2_plane=("cos2_plane","mean")
    ).sort_values(f"ctr_{dims[0]}_%", ascending=False).round(3)

In [73]:
def _origin(fig):
    fig.add_hline(y=0, line=dict(color="#bbb", width=1))
    fig.add_vline(x=0, line=dict(color="#bbb", width=1))

def plot_scree(res):
    e = res.eig.reset_index(names="axis")
    fig = go.Figure()
    fig.add_bar(x=e["axis"], y=e["%_inertia"],  name="raw %",       marker_color=PALETTE[0])
    fig.add_bar(x=e["axis"], y=e["benzecri_%"], name="Benzécri %",  marker_color=PALETTE[1])
    fig.add_scatter(x=e["axis"], y=e["cum_%"], name="cumulated raw %", yaxis="y2",
                    mode="lines+markers", line=dict(color="#555"))
    fig.update_layout(title="Scree — raw vs Benzécri-corrected inertia", barmode="group",
        yaxis_title="% of inertia", template="plotly_white", height=420,
        yaxis2=dict(title="cum %", overlaying="y", side="right", range=[0,100], showgrid=False),
        legend=dict(orientation="h", y=1.12, x=0))
    return fig

def plot_individuals(res, X, axes=(1,2), color_by=None, opacity=.75, barycentres=True):
    dx, dy = f"Dim{axes[0]}", f"Dim{axes[1]}"
    df = res.row_coord[[dx, dy]].copy()
    df["cos2"] = res.row_cos2[[dx, dy]].sum(1)
    df["ctr"]  = res.row_contrib[[dx, dy]].sum(1)
    hov = X.apply(lambda row: "<br>".join(f"{c}: <b>{v}</b>" for c, v in row.items()), axis=1)

    fig = go.Figure()
    grp = X[color_by].astype(str) if color_by else pd.Series("all", index=df.index)
    for i, g in enumerate(sorted(grp.unique())):
        m = grp == g
        fig.add_scatter(x=df.loc[m, dx], y=df.loc[m, dy], mode="markers", name=str(g),
            marker=dict(size=7, opacity=opacity, color=PALETTE[i % len(PALETTE)],
                        line=dict(width=.5, color="white")),
            customdata=np.c_[df.loc[m,"cos2"], df.loc[m,"ctr"], hov[m], df.index[m].astype(str)],
            hovertemplate="id %{customdata[3]}<br>%{customdata[2]}<br>—<br>"
                          "cos² plane %{customdata[0]:.2f} · contrib %{customdata[1]:.2f}%"
                          "<extra></extra>")
        if barycentres and color_by:      # the part you actually read
            fig.add_scatter(x=[df.loc[m,dx].mean()], y=[df.loc[m,dy].mean()],
                mode="markers+text", text=[str(g)], textposition="top center",
                marker=dict(size=16, symbol="diamond", color=PALETTE[i % len(PALETTE)],
                            line=dict(width=2, color="black")),
                showlegend=False, hoverinfo="skip")
    _origin(fig)
    fig.update_layout(title=f"Individuals{' — coloured by ' + color_by if color_by else ''}",
        xaxis_title=res.axis_label(axes[0]), yaxis_title=res.axis_label(axes[1]),
        template="plotly_white", height=620,
        yaxis=dict(scaleanchor="x", scaleratio=1), legend=dict(title=color_by or ""))
    return fig

In [74]:
def plot_modalities(res, axes=(1,2), supp=None, min_cos2=0.0, size_by="count"):
    dx, dy = f"Dim{axes[0]}", f"Dim{axes[1]}"
    d = res.col_info.copy()
    d["x"], d["y"] = res.col_coord[dx], res.col_coord[dy]
    d["ctr"]  = res.col_contrib[[dx, dy]].sum(1)
    d["cos2"] = res.col_cos2[[dx, dy]].sum(1)
    d = d[d["cos2"] >= min_cos2]
    base = d["count"] if size_by == "count" else d["ctr"]
    d["size"] = 10 + 30 * (base - base.min()) / max(base.max() - base.min(), 1e-9)

    fig = go.Figure()
    for i, (var, sub) in enumerate(d.groupby("variable", sort=False)):
        fig.add_scatter(x=sub["x"], y=sub["y"], mode="markers+text", name=var,
            text=sub["modality"], textposition="top center",
            textfont=dict(size=11, color=PALETTE[i % len(PALETTE)]),
            marker=dict(size=sub["size"], color=PALETTE[i % len(PALETTE)], opacity=.85,
                        line=dict(width=1, color="white")),
            customdata=np.c_[sub["count"], sub["ctr"], sub["cos2"]],
            hovertemplate=f"<b>%{{text}}</b> ({var})<br>n = %{{customdata[0]}}"
                          "<br>contrib plane %{customdata[1]:.1f}%"
                          "<br>cos² plane %{customdata[2]:.2f}<extra></extra>")
    if supp is not None:
        fig.add_scatter(x=supp[dx], y=supp[dy], mode="markers+text", name="supplementary",
            text=supp.index, textposition="bottom center", textfont=dict(size=11, color="#444"),
            marker=dict(size=13, symbol="x", color="#444"),
            hovertemplate="<b>%{text}</b> (illustrative)<extra></extra>")
    _origin(fig)
    fig.update_layout(title="Modalities — size ∝ " +
        ("frequency" if size_by == "count" else "contribution"),
        xaxis_title=res.axis_label(axes[0]), yaxis_title=res.axis_label(axes[1]),
        template="plotly_white", height=640,
        yaxis=dict(scaleanchor="x", scaleratio=1), legend=dict(title="variable"))
    return fig

def plot_quality_circle(res, axes=(1,2), label_above=.3):
    """Correlation-circle analogue. A modality sits at
    (sign(F1)*sqrt(cos²1), sign(F2)*sqrt(cos²2)), so its radius IS sqrt(cos² of
    the plane): on the outer ring = perfectly represented, near the centre = do
    not interpret its position on the factor map."""
    dx, dy = f"Dim{axes[0]}", f"Dim{axes[1]}"
    d = res.col_info.copy()
    d["x"] = np.sign(res.col_coord[dx]) * np.sqrt(res.col_cos2[dx])
    d["y"] = np.sign(res.col_coord[dy]) * np.sqrt(res.col_cos2[dy])
    d["q"] = np.sqrt(res.col_cos2[[dx, dy]].sum(1))

    fig, t = go.Figure(), np.linspace(0, 2*np.pi, 200)
    for rad, dash, col in [(1,"solid","#333"), (.7,"dot","#999"), (.5,"dot","#ccc")]:
        fig.add_scatter(x=rad*np.cos(t), y=rad*np.sin(t), mode="lines",
                        line=dict(color=col, dash=dash, width=1),
                        hoverinfo="skip", showlegend=False)
    for i, (var, sub) in enumerate(d.groupby("variable", sort=False)):
        fig.add_scatter(x=sub["x"], y=sub["y"], mode="markers+text", name=var,
            text=[m if q > label_above else "" for m, q in zip(sub["modality"], sub["q"])],
            textposition="top center", textfont=dict(size=10),
            marker=dict(size=11, color=PALETTE[i % len(PALETTE)],
                        line=dict(width=1, color="white")),
            customdata=np.c_[sub["modality"], sub["q"], sub["count"]],
            hovertemplate=f"<b>%{{customdata[0]}}</b> ({var})<br>"
                          "quality (√cos² plane) = %{customdata[1]:.2f}"
                          "<br>n = %{customdata[2]}<extra></extra>")
    _origin(fig)
    fig.update_layout(title="Quality of representation — closer to the outer ring = better",
        xaxis=dict(title=f"√cos² Dim{axes[0]} (signed)", range=[-1.15,1.15], zeroline=False),
        yaxis=dict(title=f"√cos² Dim{axes[1]} (signed)", range=[-1.15,1.15],
                   scaleanchor="x", scaleratio=1, zeroline=False),
        template="plotly_white", height=640, legend=dict(title="variable"))
    return fig

def plot_contribution_bars(res, axis=1, k=15):
    d = top_contributions(res, axis, k).iloc[::-1]
    fig = go.Figure(go.Bar(
        x=d["contrib_%"], y=d["modality"] + "  (" + d["variable"] + ")", orientation="h",
        marker_color=np.where(d["coord"] >= 0, PALETTE[0], PALETTE[3]),
        customdata=np.c_[d["coord"], d["cos2"], d["count"]],
        hovertemplate="contrib %{x:.1f}%<br>coord %{customdata[0]:.2f}"
                      "<br>cos² %{customdata[1]:.2f}<br>n = %{customdata[2]}<extra></extra>"))
    fig.add_vline(x=100/res.K, line=dict(color="red", dash="dash"),
                  annotation_text="uniform threshold (100/K)")
    fig.update_layout(title=f"Top contributions to Dim{axis} — blue = + pole, red = − pole",
        xaxis_title="contribution (%)", template="plotly_white",
        height=max(360, 26*len(d)), margin=dict(l=220))
    return fig

In [84]:
res  = MCA(X_active, n_components=5)

display(res.eig.round(3))
display(top_contributions(res, axis=1, k=10))
display(top_cos2(res, axes=(1,2)))
display(variable_summary(res))

,eigenvalue,%_inertia,cum_%,benzecri_eig,benzecri_%,benzecri_cum_%
Dim1,0.431,8.926,8.926,0.101,37.344,37.344
Dim2,0.356,7.373,16.299,0.052,19.169,56.513
Dim3,0.324,6.705,23.004,0.036,13.202,69.715
Dim4,0.258,5.336,28.341,0.012,4.437,74.152
Dim5,0.245,5.070,33.411,0.009,3.275,77.427


,variable,modality,count,coord,contrib_%,cos2,pole,above_avg
exclusivity_exclusive,exclusivity,exclusive,212,1.335,16.181,0.548,+,True
status_married,status,married,147,1.388,12.123,0.375,+,True
exclusivity_uncommitted,exclusivity,uncommitted,279,-0.961,11.045,0.414,-,True
reciprocity_one_sided,reciprocity,one_sided,207,-1.058,9.925,0.333,-,True
status_seduction,status,seduction,133,-1.235,8.681,0.264,-,True
reciprocity_reciprocal,reciprocity,reciprocal,232,0.738,5.416,0.189,+,True
framing_positive,framing,positive,89,1.189,5.386,0.155,+,True
outcome_separation_erasure,outcome,separation_erasure,323,-0.565,4.412,0.178,-,True
framing_predatory_or_corrupting,framing,predatory_or_corrupting,74,-1.077,3.679,0.104,-,True
outcome_stable_or_open,outcome,stable_or_open,184,0.641,3.240,0.105,+,True


,variable,modality,count,cos2_Dim1,cos2_Dim2,cos2_plane,quality
exclusivity_exclusive,exclusivity,exclusive,212,0.548,0.105,0.653,0.808
status_married,status,married,147,0.375,0.130,0.505,0.710
exclusivity_uncommitted,exclusivity,uncommitted,279,0.414,0.017,0.431,0.657
status_ambiguous_friendship,status,ambiguous_friendship,131,0.000,0.422,0.422,0.650
relation_type_female_male,relation_type,female_male,697,0.014,0.355,0.369,0.608
reciprocity_one_sided,reciprocity,one_sided,207,0.333,0.007,0.340,0.583
exclusivity_unclear,exclusivity,unclear,169,0.000,0.297,0.297,0.545
status_seduction,status,seduction,133,0.264,0.021,0.285,0.533
reciprocity_reciprocal,reciprocity,reciprocal,232,0.189,0.048,0.237,0.487
framing_decadent_sublime,framing,decadent_sublime,166,0.015,0.188,0.203,0.451


,n_modalities,ctr_Dim1_%,ctr_Dim2_%,mean_cos2_plane
variable,,,,
exclusivity,5,27.583,20.706,0.304
status,7,26.482,30.443,0.227
reciprocity,4,18.385,3.479,0.183
outcome,7,14.073,8.737,0.097
framing,8,11.829,17.713,0.097
relation_type,4,1.647,18.921,0.192


In [76]:
plot_individuals(res, X_active).show()

In [90]:
fig = plot_modalities(res, min_cos2=0.05)
fig.show()
# save figure in high resolution
fig.write_image(IMGS_PATH / "mca_modalities.png", scale=2, width=1000, height=800)

In [46]:
plot_quality_circle(res).show()
plot_contribution_bars(res, axis=1).show()
plot_contribution_bars(res, axis=2).show()